# Модель для микродрона: человек и красное окно

Обучает YOLO11n и переводит её в формат `.cvimodel` для бортового компьютера микродрона «Сверх».
Сделано по ноутбуку производителя `yolo_train_and_export.ipynb`: команды конвертации те же,
настройки собраны в одну ячейку.

Запускать в Google Colab с GPU: **Среда выполнения → Сменить среду выполнения → T4 GPU**,
затем **Выполнить все**. Результат — файл `<model_name>.cvimodel` на Google Диске.

## 1. Настройки — менять только здесь

In [ ]:
model_name = "rescue"          # имя модели: получится rescue.cvimodel
roboflow_url = "https://app.roboflow.com/ds/КЛЮЧ"   # ссылка на датасет: Roboflow → Download dataset → YOLOv11 → show download code → Raw URL
img_size = 640                 # вход модели: 640 точнее, 320 примерно вдвое быстрее на дроне
epochs = 300                   # обучение остановится раньше, если качество перестанет расти
patience = 50
calibration_images = 200       # кадров для калибровки int8 (из обучающей выборки)

## 2. Подготовка

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip3 install -q ultralytics

## 3. Датасет

Классы в датасете: `person` (манекены, ростовые фигуры, плакаты с человеком — часть наполовину
закрыта предметами) и `window` (красное окно). Снимайте камерой самого микродрона, на полигоне или
при похожем свете.

In [ ]:
!mkdir -p /content/dataset
%cd /content/dataset
!curl -L "$roboflow_url" > roboflow.zip; unzip -q -o roboflow.zip; rm roboflow.zip
%cd /content

In [ ]:
import yaml
yaml_path = '/content/dataset/data.yaml'
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data['train'] = '/content/dataset/train'
data['val'] = '/content/dataset/valid'
data['test'] = '/content/dataset/test'
with open(yaml_path, 'w') as f:
    yaml.safe_dump(data, f)
class_names = data['names'] if isinstance(data['names'], list) else list(data['names'].values())
print('Классы:', class_names)
!mkdir -p /content/drive/MyDrive/sverk_train

## 4. Обучение

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
results = model.train(data="/content/dataset/data.yaml", epochs=epochs, patience=patience, imgsz=img_size,
                      save_period=10, project="/content/drive/MyDrive/sverk_train", name=model_name, exist_ok=True)

Графики и метрики — в `/content/drive/MyDrive/sverk_train/<model_name>/`. Ориентир из документации:
mAP50 выше 0,7 — хорошо. Таблица метрик пригодится для отчёта (аспекты A1.3, A2.6).

In [ ]:
import shutil
best = f"/content/drive/MyDrive/sverk_train/{model_name}/weights/best.pt"
shutil.copy(best, f"/content/{model_name}.pt")
shutil.copy(best, f"/content/drive/MyDrive/{model_name}.pt")
metrics = YOLO(best).val(data="/content/dataset/data.yaml", imgsz=img_size)
print("mAP50:", round(metrics.box.map50, 3), " mAP50-95:", round(metrics.box.map, 3))

## 5. Инструменты конвертации (долго, 5–10 минут)

In [ ]:
%cd /content
!pip3 install -q gdown
!gdown https://drive.google.com/uc?id=1DjNdqC0Gd5AUYzCC0clU5GAY2xRLxny2 -O tpu-mlir.tar.gz
!mkdir -p tpu_mlir
!tar -xzf tpu-mlir.tar.gz -C tpu_mlir --strip-components=1
!mv ./tpu_mlir/$(ls ./tpu_mlir) ./tpu_mlir/source
!wget -q https://github.com/sophgo/tpu-mlir/releases/download/v1.14/tpu_mlir-1.14-py3-none-any.whl
!sudo add-apt-repository ppa:deadsnakes/ppa -y
!sudo apt update
!sudo apt install -y python3.10
!wget -q https://bootstrap.pypa.io/get-pip.py
!python3.10 get-pip.py
!python3.10 -m pip install tpu_mlir-1.14-py3-none-any.whl
!python3.10 -m pip install flatbuffers ultralytics onnxsim matplotlib-inline onnxruntime
!wget -q https://raw.githubusercontent.com/ret7020/LicheeRVNano/refs/heads/master/Projects/Yolov8/export_to_onnx.py
!python3.10 -m pip install onnx==1.13.1

## 6. Экспорт в ONNX и конвертация в cvimodel

In [ ]:
import os
os.environ['MODEL_NAME'] = model_name
os.environ['IMG_SIZE_W'] = str(img_size)
os.environ['IMG_SIZE_H'] = str(img_size)
os.environ['CALIBRATION_DATASET_PATH'] = "/content/dataset/train/images/"
os.environ['CALIBRATION_IMAGES_COUNT'] = str(calibration_images)
os.environ['TEST_IMG'] = os.environ['CALIBRATION_DATASET_PATH'] + sorted(os.listdir(os.environ['CALIBRATION_DATASET_PATH']))[0]
!python3.10 export_to_onnx.py /content/$MODEL_NAME.pt $IMG_SIZE_W $IMG_SIZE_H
!mkdir -p /content/work
%cd /content/work

In [ ]:
!export PYTHONPATH=/usr/bin/python3.10
!source ../tpu_mlir/source/envsetup.sh
!model_transform.py --model_name $MODEL_NAME --model_def "/content/${MODEL_NAME}.onnx" --input_shapes [[1,3,$IMG_SIZE_W,$IMG_SIZE_H]] --mean 0.0,0.0,0.0 --scale 0.0039216,0.0039216,0.0039216 --keep_aspect_ratio --pixel_format rgb --test_input $TEST_IMG --test_result "${MODEL_NAME}_top_outputs.npz" --mlir $MODEL_NAME.mlir

In [ ]:
!source ../tpu_mlir/source/envsetup.sh
!run_calibration.py $MODEL_NAME.mlir \
  --dataset $CALIBRATION_DATASET_PATH \
  --input_num $CALIBRATION_IMAGES_COUNT \
  -o "${MODEL_NAME}_cali_table"

In [ ]:
!mkdir -p result
!source ../tpu_mlir/source/envsetup.sh
!model_deploy.py \
  --mlir $MODEL_NAME.mlir \
  --quant_input \
  --quant_output \
  --quantize int8 \
  --calibration_table "${MODEL_NAME}_cali_table" \
  --processor cv181x \
  --tolerance 0.85,0.45 \
  --model "./result/${MODEL_NAME}.cvimodel"

## 7. Готово: модель на Диск и памятка для дрона

In [ ]:
out = f"/content/work/result/{model_name}.cvimodel"
assert os.path.exists(out), "cvimodel не собрался — смотрите ошибки в ячейках выше"
shutil.copy(out, "/content/drive/MyDrive/")
print(f"Модель: Google Диск / {model_name}.cvimodel, {os.path.getsize(out) / 1e6:.1f} МБ")
print()
print("На дрон (по проводу с Linux/macOS; с Windows — 10.10.1.1):")
print(f"  scp {model_name}.cvimodel root@10.10.0.1:/root/")
print("В /root/config.yaml на дроне (образец — /root/config_sample.yaml):")
print(f"  model.path: /root/{model_name}.cvimodel")
print(f"  model.class_cnt: {len(class_names)}")
print(f"  classes.names: {class_names}")
print("Запуск на дроне:")
print("  killall face_recognizer; /usr/opt/tiny_control/yolo_detector /root/config.yaml")